## 과제 2

생성형 문장에 대한 평가는 아직도 명확한 방법이 정해지지 않았습니다.  
정량적, 정성적 평가 지표를 설계하십시오.  
아래는 활용할 데이터를 받아올 수 있는 링크입니다.
(아래)
https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&dataSetSn=71849

### 생성형 문장 평가 지표 설계

생성형 문장은 하나의 정답만 존재하지 않는다. 같은 영상을 설명하더라도 표현 방식, 문장 순서, 세부 묘사는 달라질 수 있다. 따라서 생성 문장 평가는 정답 문장과의 단순 일치율만으로 판단하기 어렵고, 자동화된 정량 평가와 사람이 직접 판단하는 정성 평가를 함께 사용해야 한다.

제시된 AI-Hub 데이터는 영상 또는 이미지 장면에 대한 설명문 생성에 활용할 수 있는 데이터이므로, 평가는 다음 세 가지를 중심으로 설계한다.

1. 생성 문장이 기준 설명문과 의미적으로 유사한가
2. 생성 문장이 실제 영상 내용과 일치하는가
3. 문장이 자연스럽고 구체적이며 활용 가능한가

### 1. 정량적 평가 지표

#### 1.1 참조 문장 기반 유사도

정답 설명문과 생성 문장 사이의 유사도를 계산한다. 단, 생성형 문장은 다양한 정답이 가능하므로 단순 n-gram 일치율만 사용하지 않고 의미 기반 지표를 함께 사용한다.

| 지표 | 평가 내용 | 장점 | 한계 |
|---|---|---|---|
| BLEU | 생성 문장과 정답 문장의 n-gram 일치율 | 계산이 간단함 | 표현이 다르면 낮게 평가됨 |
| ROUGE-L | 가장 긴 공통 부분 수열 기반 유사도 | 문장 구조 유사도 반영 | 의미적 동일성을 충분히 반영하지 못함 |
| METEOR | 단어 일치, 어간, 동의어 등을 고려 | BLEU보다 의미 유사도 반영이 좋음 | 한국어에서는 형태소 분석 품질에 영향받음 |
| BERTScore | 문장 임베딩 기반 의미 유사도 | 표현이 달라도 의미가 같으면 높은 점수 가능 | 모델 선택에 따라 점수가 달라질 수 있음 |

참조 문장 유사도 점수는 다음과 같이 계산할 수 있다.

```text
Reference Similarity Score
= 0.15 * BLEU
+ 0.20 * ROUGE-L
+ 0.30 * METEOR
+ 0.35 * BERTScore
```

BLEU의 비중을 낮추고 METEOR, BERTScore의 비중을 높인 이유는 생성형 문장의 경우 정답과 다른 표현도 정답으로 인정될 수 있기 때문이다.

#### 1.2 영상-문장 정합도

생성 문장이 실제 영상 또는 이미지의 내용과 맞는지 평가한다. 기준 문장과 비슷하더라도 영상에 없는 내용을 생성하면 좋은 답변으로 볼 수 없다.

예를 들어 영상에는 산과 강만 있는데 생성 문장이 “사람들이 도심에서 쇼핑하고 있다”라고 하면 문장은 자연스러워도 잘못된 생성이다.

활용 가능한 지표는 다음과 같다.

```text
Visual-Text Matching Score
= cosine_similarity(영상 임베딩, 생성 문장 임베딩)
```

CLIP, VideoCLIP, BLIP 계열 모델을 활용하여 영상 프레임과 생성 문장의 임베딩 유사도를 계산할 수 있다.

#### 1.3 핵심 정보 포함률

생성 문장이 장면 설명에 필요한 핵심 정보를 얼마나 포함하는지 평가한다. 영상 설명문에서는 객체, 배경, 행위, 분위기, 시간적 흐름 등이 중요하다.

평가 항목 예시는 다음과 같다.

| 항목 | 설명 |
|---|---|
| 객체 | 영상에 등장하는 주요 사물, 건물, 자연물 등을 언급했는가 |
| 배경 | 장소나 공간적 배경을 설명했는가 |
| 행위 | 인물이나 사물의 움직임을 설명했는가 |
| 분위기 | 장면의 정서, 날씨, 조명, 색감 등을 설명했는가 |
| 시간성 | 영상의 변화나 흐름을 반영했는가 |

```text
Attribute Coverage Score
= 생성 문장에 포함된 핵심 속성 수 / 전체 핵심 속성 수
```

예를 들어 핵심 속성이 객체, 배경, 행위, 분위기, 시간성 5개이고 생성 문장이 이 중 4개를 포함하면 점수는 0.8이다.

#### 1.4 환각률

환각은 생성형 모델이 실제 데이터에 없는 내용을 사실처럼 만들어내는 문제이다. 영상 설명문 생성에서는 매우 중요한 감점 요소이다.

```text
Hallucination Rate
= 영상에 존재하지 않는 내용의 언급 수 / 생성 문장의 전체 주요 언급 수
```

환각률이 높을수록 생성 품질은 낮다. 최종 점수에는 다음과 같이 반영할 수 있다.

```text
Faithfulness Score = 1 - Hallucination Rate
```

#### 1.5 언어 품질 점수

생성 문장이 문법적으로 자연스럽고 읽기 쉬운지도 평가한다.

| 지표 | 설명 |
|---|---|
| 문법 오류율 | 맞춤법, 띄어쓰기, 비문 여부 |
| 반복률 | 같은 단어 또는 구절의 불필요한 반복 여부 |
| 길이 적절성 | 설명문이 지나치게 짧거나 장황하지 않은지 |
| 어휘 다양성 | 동일 표현만 반복하지 않고 다양한 표현을 사용하는지 |

```text
Repetition Rate
= 반복 n-gram 수 / 전체 n-gram 수
```

```text
Length Adequacy Score
= 기준 길이 범위에 포함되면 1, 너무 짧거나 길면 감점
```

### 2. 정성적 평가 지표

정성 평가는 사람이 직접 문장을 읽고 판단하는 방식이다. 자동 지표가 잡아내기 어려운 문맥, 자연스러움, 구체성, 실제 활용 가능성을 평가할 수 있다. 각 항목은 1점부터 5점까지 부여한다.

| 평가 항목 | 평가 기준 | 점수 |
|---|---|---|
| 사실성 | 생성 문장이 실제 영상 내용과 일치하는가 | 1~5점 |
| 충실성 | 영상의 핵심 장면을 빠뜨리지 않고 설명했는가 | 1~5점 |
| 구체성 | 객체, 배경, 분위기, 행위 등을 구체적으로 묘사했는가 | 1~5점 |
| 자연스러움 | 한국어 문장이 어색하지 않고 읽기 쉬운가 | 1~5점 |
| 일관성 | 문장 전체가 모순 없이 연결되는가 | 1~5점 |
| 간결성 | 불필요한 반복이나 장황한 표현이 없는가 | 1~5점 |
| 활용성 | 검색, 요약, 접근성 설명 등에 실제로 사용할 수 있는가 | 1~5점 |

정성 평가 점수는 다음과 같이 계산한다.

```text
Qualitative Score
= 0.25 * 사실성
+ 0.20 * 충실성
+ 0.15 * 구체성
+ 0.15 * 자연스러움
+ 0.10 * 일관성
+ 0.05 * 간결성
+ 0.10 * 활용성
```

사실성과 충실성에 가장 높은 가중치를 둔다. 생성 문장이 아무리 자연스러워도 실제 영상과 맞지 않으면 좋은 설명문이라고 보기 어렵기 때문이다.

### 3. 최종 통합 평가 지표

최종 평가는 정량 평가와 정성 평가를 결합한다.

```text
Quantitative Score
= 0.30 * Reference Similarity Score
+ 0.30 * Visual-Text Matching Score
+ 0.20 * Attribute Coverage Score
+ 0.15 * Faithfulness Score
+ 0.05 * Language Quality Score
```

```text
Final Score
= 0.40 * Quantitative Score
+ 0.60 * Qualitative Score
```

정성 평가의 비중을 더 높게 둔 이유는 생성형 문장의 품질이 단순 수치로 완전히 판단되기 어렵기 때문이다. 특히 영상 설명문에서는 장면의 맥락, 분위기, 정보의 중요도 판단이 필요하므로 사람 평가가 반드시 포함되어야 한다.

### 4. 평가 예시

생성 문장 예시:

```text
영상에는 강을 따라 이어진 산책로와 주변의 나무들이 보이며, 맑은 날씨 속에서 도시 외곽의 평온한 풍경이 나타난다. 화면은 천천히 이동하며 강변의 구조물과 녹지를 함께 보여준다.
```

이 문장은 실제 영상에 강, 산책로, 나무, 도시 외곽 풍경이 존재한다면 높은 점수를 받을 수 있다. 그러나 영상에 사람이 없는데 “사람들이 산책하고 있다”라고 표현하면 환각으로 감점한다.

### 5. 결론

생성형 문장 평가는 하나의 지표만으로 판단할 수 없다. BLEU, ROUGE 같은 기존 자동 지표는 정답 문장과의 표면적 유사도는 측정할 수 있지만, 실제 영상과의 일치성이나 문장의 자연스러움까지 충분히 평가하지 못한다.

따라서 본 과제에서는 다음과 같은 다층 평가 체계를 제안한다.

1. 기준 설명문과의 의미 유사도 평가
2. 영상과 생성 문장의 정합도 평가
3. 핵심 객체와 배경 정보 포함률 평가
4. 영상에 없는 내용을 생성했는지 확인하는 환각률 평가
5. 사람이 직접 판단하는 사실성, 충실성, 자연스러움 평가

이와 같이 정량적 평가와 정성적 평가를 함께 사용하면 생성형 문장의 정확성, 자연스러움, 활용 가능성을 균형 있게 판단할 수 있다.


### 평가 지표 실행 코드

아래 코드는 `reference`와 `generated` 컬럼이 있는 데이터프레임에 대해 생성형 문장 평가 지표를 계산한다. 별도 파일이 없을 때도 예시 데이터로 바로 실행된다.


In [1]:
from collections import Counter
from difflib import SequenceMatcher
import math
import re
from pathlib import Path

import pandas as pd


def normalize_text(text):
    """Lowercase and remove repeated spaces."""
    text = '' if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r'[^0-9a-zA-Z가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize(text):
    return normalize_text(text).split()


def ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def bleu_score(reference, generated, max_n=4):
    ref_tokens = tokenize(reference)
    gen_tokens = tokenize(generated)
    if not ref_tokens or not gen_tokens:
        return 0.0

    precisions = []
    for n in range(1, max_n + 1):
        gen_counts = Counter(ngrams(gen_tokens, n))
        ref_counts = Counter(ngrams(ref_tokens, n))
        if not gen_counts:
            precisions.append(0.0)
            continue
        overlap = sum(min(count, ref_counts[gram]) for gram, count in gen_counts.items())
        precisions.append((overlap + 1) / (sum(gen_counts.values()) + 1))

    log_precision = sum(math.log(p) for p in precisions) / max_n
    brevity_penalty = 1.0 if len(gen_tokens) > len(ref_tokens) else math.exp(1 - len(ref_tokens) / len(gen_tokens))
    return float(brevity_penalty * math.exp(log_precision))


def rouge_l_score(reference, generated):
    ref_tokens = tokenize(reference)
    gen_tokens = tokenize(generated)
    if not ref_tokens or not gen_tokens:
        return 0.0

    matcher = SequenceMatcher(None, ref_tokens, gen_tokens)
    lcs = sum(block.size for block in matcher.get_matching_blocks())
    recall = lcs / len(ref_tokens)
    precision = lcs / len(gen_tokens)
    if recall + precision == 0:
        return 0.0
    return float(2 * recall * precision / (recall + precision))


def meteor_like_score(reference, generated):
    ref_tokens = tokenize(reference)
    gen_tokens = tokenize(generated)
    if not ref_tokens or not gen_tokens:
        return 0.0

    ref_counts = Counter(ref_tokens)
    gen_counts = Counter(gen_tokens)
    overlap = sum(min(count, ref_counts[token]) for token, count in gen_counts.items())
    precision = overlap / len(gen_tokens)
    recall = overlap / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return float((10 * precision * recall) / (recall + 9 * precision))


def extract_keywords(text, min_len=2):
    tokens = tokenize(text)
    stopwords = {
        'the', 'a', 'an', 'and', 'or', 'is', 'are', 'was', 'were', 'to', 'of', 'in', 'on', 'for',
        'with', 'this', 'that', 'it', 'as', 'by', 'be', '있다', '있는', '한다', '그리고', '또는'
    }
    return {token for token in tokens if len(token) >= min_len and token not in stopwords}


def attribute_coverage_score(reference, generated):
    ref_keywords = extract_keywords(reference)
    gen_keywords = extract_keywords(generated)
    if not ref_keywords:
        return 0.0
    return len(ref_keywords & gen_keywords) / len(ref_keywords)


def hallucination_rate(reference, generated):
    ref_keywords = extract_keywords(reference)
    gen_keywords = extract_keywords(generated)
    if not gen_keywords:
        return 0.0
    return len(gen_keywords - ref_keywords) / len(gen_keywords)


def repetition_rate(generated, n=2):
    gen_tokens = tokenize(generated)
    grams = ngrams(gen_tokens, n)
    if not grams:
        return 0.0
    counts = Counter(grams)
    repeated = sum(count - 1 for count in counts.values() if count > 1)
    return repeated / len(grams)


def length_adequacy_score(generated, min_tokens=8, max_tokens=120):
    length = len(tokenize(generated))
    if min_tokens <= length <= max_tokens:
        return 1.0
    if length == 0:
        return 0.0
    if length < min_tokens:
        return length / min_tokens
    return max(0.0, 1 - (length - max_tokens) / max_tokens)


def language_quality_score(generated):
    repeat_penalty = repetition_rate(generated)
    length_score = length_adequacy_score(generated)
    return max(0.0, 0.7 * length_score + 0.3 * (1 - repeat_penalty))


def evaluate_one(reference, generated):
    bleu = bleu_score(reference, generated)
    rouge_l = rouge_l_score(reference, generated)
    meteor = meteor_like_score(reference, generated)
    coverage = attribute_coverage_score(reference, generated)
    hallucination = hallucination_rate(reference, generated)
    faithfulness = 1 - hallucination
    lang_quality = language_quality_score(generated)

    reference_similarity = 0.15 * bleu + 0.20 * rouge_l + 0.30 * meteor + 0.35 * coverage
    quantitative = (
        0.30 * reference_similarity
        + 0.20 * coverage
        + 0.30 * faithfulness
        + 0.20 * lang_quality
    )

    return {
        'bleu': bleu,
        'rouge_l': rouge_l,
        'meteor_like': meteor,
        'attribute_coverage': coverage,
        'hallucination_rate': hallucination,
        'faithfulness': faithfulness,
        'language_quality': lang_quality,
        'reference_similarity': reference_similarity,
        'quantitative_score': quantitative,
    }


def evaluate_dataframe(df, reference_col='reference', generated_col='generated'):
    rows = []
    for _, row in df.iterrows():
        rows.append(evaluate_one(row[reference_col], row[generated_col]))
    metric_df = pd.DataFrame(rows)
    return pd.concat([df.reset_index(drop=True), metric_df], axis=1)


sample_df = pd.DataFrame({
    'reference': [
        '영상에는 강을 따라 이어진 산책로와 주변의 나무들이 보이며 맑은 날씨 속에서 평온한 도시 외곽 풍경이 나타난다.',
        '무대 위에서 기타를 연주하는 사람이 보이고 조명이 어두운 공연장 분위기를 만든다.'
    ],
    'generated': [
        '강변 산책로와 나무가 보이고 맑은 날씨의 조용한 도시 외곽 풍경이 나타난다.',
        '사람들이 해변에서 축구를 하며 밝은 햇살 아래 즐거운 시간을 보낸다.'
    ]
})

result_df = evaluate_dataframe(sample_df)
result_df.round(4)


,reference,generated,bleu,rouge_l,meteor_like,attribute_coverage,hallucination_rate,faithfulness,language_quality,reference_similarity,quantitative_score
0,영상에는 강을 따라 이어진 산책로와 주변의 나무들이 보이며 맑은 날씨 속에서 평온한...,강변 산책로와 나무가 보이고 맑은 날씨의 조용한 도시 외곽 풍경이 나타난다.,0.2189,0.4444,0.3871,0.375,0.4545,0.5455,1.0,0.3691,0.5494
1,무대 위에서 기타를 연주하는 사람이 보이고 조명이 어두운 공연장 분위기를 만든다.,사람들이 해변에서 축구를 하며 밝은 햇살 아래 즐거운 시간을 보낸다.,0.0959,0.0000,0.0000,0.000,1.0000,0.0000,1.0,0.0144,0.2043


In [2]:
# reference/generated 컬럼이 있는 CSV 또는 JSONL 파일을 평가할 때 사용한다.
# 예: eval_df = load_generation_file('my_generation_result.csv')
#     scored_df = evaluate_dataframe(eval_df)
#     scored_df.to_csv('generation_eval_result.csv', index=False, encoding='utf-8-sig')

def load_generation_file(path):
    path = Path(path)
    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path)
    elif path.suffix.lower() in {'.jsonl', '.json'}:
        df = pd.read_json(path, lines=path.suffix.lower() == '.jsonl')
    else:
        raise ValueError('csv, jsonl, json 파일만 지원합니다.')

    required = {'reference', 'generated'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'필수 컬럼이 없습니다: {sorted(missing)}')
    return df


qualitative_template = pd.DataFrame({
    'item': ['factuality', 'faithfulness', 'specificity', 'naturalness', 'coherence', 'conciseness', 'usefulness'],
    'description': [
        '실제 영상/원문 내용과 일치하는가',
        '핵심 정보를 빠뜨리지 않았는가',
        '객체, 배경, 행위, 분위기를 구체적으로 설명했는가',
        '문장이 자연스럽고 읽기 쉬운가',
        '문장 전체가 모순 없이 연결되는가',
        '불필요한 반복이나 장황함이 없는가',
        '검색, 요약, 설명 용도로 활용 가능한가',
    ],
    'weight': [0.25, 0.20, 0.15, 0.15, 0.10, 0.05, 0.10],
    'score_1_to_5': [None] * 7,
})

qualitative_template


,item,description,weight,score_1_to_5
0,factuality,실제 영상/원문 내용과 일치하는가,0.25,None
1,faithfulness,핵심 정보를 빠뜨리지 않았는가,0.20,None
2,specificity,"객체, 배경, 행위, 분위기를 구체적으로 설명했는가",0.15,None
3,naturalness,문장이 자연스럽고 읽기 쉬운가,0.15,None
4,coherence,문장 전체가 모순 없이 연결되는가,0.10,None
5,conciseness,불필요한 반복이나 장황함이 없는가,0.05,None
6,usefulness,"검색, 요약, 설명 용도로 활용 가능한가",0.10,None
